# Social Blade Scraper — Instagram · TikTok · YouTube

In [ ]:
# !pip install -q cloudscraper beautifulsoup4 pandas openpyxl

In [ ]:
import cloudscraper
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime


def parse_number(text):
    """Converteix '1.2K', '3.5M', '6.4%', '12,345' a int/float."""
    if not text:
        return None
    text = text.strip().replace(",", "")
    if text.endswith("%"):
        try:
            return float(text[:-1])
        except ValueError:
            return text
    multipliers = {"K": 1_000, "M": 1_000_000, "B": 1_000_000_000}
    for suffix, mult in multipliers.items():
        if text.upper().endswith(suffix):
            return int(float(text[:-1]) * mult)
    try:
        return int(float(text))
    except ValueError:
        return text


def scrape_socialblade(url):
    """
    Scraper genèric per a qualsevol pàgina de Social Blade.
    Retorna un dict amb tots els KPIs del grid de capçalera + history_df.
    """
    scraper = cloudscraper.create_scraper(
        browser={"browser": "chrome", "platform": "darwin", "mobile": False}
    )
    resp = scraper.get(url, timeout=20)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")
    data = {"url": url, "scraped_at": datetime.now().isoformat()}

    # Grid de KPIs: <div class="grid lg:hidden ...">
    kpi_grid = soup.find("div", class_=lambda c: c and "lg:hidden" in c and "grid" in c)
    if kpi_grid:
        for item in kpi_grid.find_all("div"):
            ps = item.find_all("p")
            if len(ps) == 2:
                key = ps[0].get_text(strip=True).lower().replace(" ", "_")
                data[key] = parse_number(ps[1].get_text(strip=True))
    else:
        print("⚠️  Grid de KPIs no trobat")

    # Taula d'historial
    table = soup.find("table", {"id": "DataTables_Table_0"}) or soup.find("table")
    if table:
        headers = [th.get_text(strip=True) for th in table.find_all("th")]
        rows_data = [
            [td.get_text(strip=True) for td in tr.find_all("td")]
            for tr in table.find_all("tr")[1:]
            if tr.find_all("td")
        ]
        if rows_data:
            n_cols = len(rows_data[0])
            while len(headers) < n_cols:
                headers.append(f"col_{len(headers)}")
            data["history_df"] = pd.DataFrame(rows_data, columns=headers[:n_cols])

    return data


def display_result(data):
    print(f"  URL: {data['url']}")
    print(f"  Scraped at: {data['scraped_at']}")
    print("-" * 40)
    for k, v in data.items():
        if k in ("url", "scraped_at", "history_df"):
            continue
        val = f"{v:,}" if isinstance(v, int) else f"{v:.2f}" if isinstance(v, float) else v
        print(f"  {k:<22} {val}")
    if "history_df" in data:
        print(f"\n  Historial: {len(data['history_df'])} files · columnes: {list(data['history_df'].columns)}")

## Instagram

In [ ]:
IG_USER = "cabrafotuda"
ig = scrape_socialblade(f"https://socialblade.com/instagram/user/{IG_USER}")
display_result(ig)

## TikTok

In [ ]:
TT_USER = "khaby.lame"
tt = scrape_socialblade(f"https://socialblade.com/tiktok/user/{TT_USER}")
display_result(tt)

## YouTube

In [ ]:
YT_HANDLE = "mrbeast"
yt = scrape_socialblade(f"https://socialblade.com/youtube/handle/{YT_HANDLE}")
display_result(yt)

## Twitch (twitchtracker.com)

In [28]:
import undetected_chromedriver as uc
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
import time


def scrape_twitchtracker(username):
    """
    Scraper per a twitchtracker.com/username/statistics
    Sense headless per bypasear el challenge de Cloudflare.
    """
    url = f"https://twitchtracker.com/{username}/statistics"

    options = uc.ChromeOptions()
    # NO headless — Cloudflare detecta i bloqueja headless
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1280,900")

    driver = uc.Chrome(options=options)
    data = {"url": url, "scraped_at": datetime.now().isoformat()}

    try:
        driver.get(url)

        # Espera fins que .pg-controls aparegui (max 20s, temps suficient pel challenge)
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CLASS_NAME, "pg-controls"))
        )

        soup = BeautifulSoup(driver.page_source, "html.parser")

        pg_controls = soup.find("div", class_="pg-controls")
        if pg_controls:
            for pge in pg_controls.find_all("div", class_="pge"):
                label_tag = pge.find("div", class_="pge-t")
                value_tag = pge.find("div", class_="pge-v")
                if label_tag and value_tag:
                    span = value_tag.find("span")
                    value_text = (span or value_tag).get_text(strip=True)
                    key = label_tag.get_text(strip=True).lower().replace(" ", "_")
                    data[key] = parse_number(value_text)
        else:
            print("⚠️  Bloc .pg-controls no trobat")
    finally:
        driver.quit()

    return data


  URL: https://twitchtracker.com/simmer_valenciana/statistics
  Scraped at: 2026-03-13T12:20:23.768315
----------------------------------------
  total_followers        1,800
  hours_streamed         1,478
  average_viewers        11
  peak_viewers           72
  hours_watched          16,700
  active_days            724


In [ ]:
TW_USER = "simmer_valenciana"
tw = scrape_twitchtracker(TW_USER)
display_result(tw)

---
## Ingesta massiva per xarxa

Llegeix un Excel amb columna `user_name` i afegeix les mètriques de cada usuari al costat.

In [22]:
def ingest(df_input, scrape_fn, output_file):
    """
    Per a cada user_name del DataFrame, crida scrape_fn(username),
    afegeix les mètriques com a noves columnes i exporta a Excel.
    """
    records = []
    for username in df_input["user_name"]:
        print(f"  → {username}", end=" ")
        try:
            data = scrape_fn(str(username).strip())
            data.pop("history_df", None)
            data.pop("url", None)
            data.pop("scraped_at", None)
            data["user_name"] = username
            records.append(data)
            print("✓")
        except Exception as e:
            print(f"✗  ({e})")
            records.append({"user_name": username})

    df_out = df_input.merge(pd.DataFrame(records), on="user_name", how="left")
    df_out.to_excel(output_file, index=False)
    print(f"\nExportat → {output_file}")
    return df_out

### Instagram

In [25]:
df_ig = ingest(
    df_input=pd.read_excel("usuaris_instagram.xlsx"),
    scrape_fn=lambda u: scrape_socialblade(f"https://socialblade.com/instagram/user/{u}"),
    output_file="resultats_instagram.xlsx",
)
display(df_ig)

  → Cabrafotuda ✓
  → Larutadelsesmorzars ✓
  → frases_jordi_guerola ✓
  → cescrocastudio ✓
  → valencianismes ✓
  → Tresdeu ✓
  → Xufet_ ✓
  → apitxat ✓
  → Apala9tv ✓
  → bookcarmecarmeta_ ✓
  → valenciaalatac ✓
  → Uelamoderna ✓
  → aitana_ferrer_ ✓
  → la.sintesi ✓
  → poliedres ✓

Exportat → resultats_instagram.xlsx


,user_name,followers,following,media_count,engagement_rate,average_likes,average_comments
0,Cabrafotuda,197000,1200,881,6.40,12000,266
1,Larutadelsesmorzars,121000,285,544,1.90,2300,42
2,frases_jordi_guerola,17000,35,5400,1.50,240,5
3,cescrocastudio,25000,4800,533,8.10,2000,21
4,valencianismes,8400,62,472,14.00,1100,9
5,Tresdeu,15000,907,2200,1.50,213,6
6,Xufet_,75000,545,6400,0.50,368,1
7,apitxat,16000,1200,219,19.00,3100,54
8,Apala9tv,11000,59,1900,27.00,2900,56
9,bookcarmecarmeta_,1300,1300,225,8.10,100,5


### TikTok

In [26]:
df_tt = ingest(
    df_input=pd.read_excel("usuaris_tiktok.xlsx"),
    scrape_fn=lambda u: scrape_socialblade(f"https://socialblade.com/tiktok/user/{u}"),
    output_file="resultats_tiktok.xlsx",
)
display(df_tt)

  → cabrafotuda ✓
  → apitxat ✓
  → misstagless ✓
  → mariamandarinaa ✓
  → _noemtoqueslallengua_ ✓
  → Redeula_ ✓
  → Socundinosaure ✓
  → coneixement11 ✓
  → clautellsyou ✓
  → antonio_iau ✓
  → daviddelsud ✓
  → bancaldoliveres ✓
  → Lletivi22 ✓
  → cescroca ✓
  → simmervalenciana ✓

Exportat → resultats_tiktok.xlsx


,user_name,followers,following,likes,videos,created_on
0,cabrafotuda,232000,319,8100000,404,Jul 16 1971
1,apitxat,83000,221,3000000,335,Aug 25 2020
2,misstagless,10000,935,218000,242,Jan 5 2021
3,mariamandarinaa,5000,378,195000,400,Oct 10 1971
4,_noemtoqueslallengua_,2100,196,39000,207,Mar 19 2021
5,Redeula_,3000,731,61000,352,Mar 22 2020
6,Socundinosaure,2400,504,116000,1400,May 10 2020
7,coneixement11,1700,334,18000,57,Feb 17 2021
8,clautellsyou,1700,514,89000,97,Jan 16 2020
9,antonio_iau,2000,102,20000,124,Mar 21 2020


### Twitch

In [ ]:
df_tw = ingest(
    df_input=pd.read_excel("usuaris_twitch.xlsx"),
    scrape_fn=scrape_twitchtracker,
    output_file="resultats_twitch.xlsx",
)
display(df_tw)

### YouTube

In [ ]:
df_yt = ingest(
    df_input=pd.read_excel("usuaris_youtube.xlsx"),
    scrape_fn=lambda u: scrape_socialblade(f"https://socialblade.com/youtube/handle/{u}"),
    output_file="resultats_youtube.xlsx",
)
display(df_yt)